# 07 · Advanced · Write a scanner class

Subclass `Scanner` for full control over evidence and confidence. The lifecycle is
`discover → analyze → score → recommend`; `score` is inherited. The framework still
applies exclusions, validates every step's output, applies the confidence gate, and
scores the results.

In [ ]:
%pip install /Volumes/main/default/lakehouse_gc/lakehouse_gc-0.2.2-py3-none-any.whl

In [ ]:
from collections.abc import Sequence
from dataclasses import dataclass

from lakehouse_gc import rules_dataframe, run
from lakehouse_gc.core import (
    Dimension,
    Evidence,
    EvidenceStatus,
    Finding,
    Outcome,
    Recommendation,
    Resource,
    ResourceRef,
    RuleEvaluation,
    RuleSpec,
    Scanner,
    ScannerContext,
    ScoredEvaluation,
    Severity,
)

POLICY_PATH = "/Volumes/main/default/lakehouse_gc/policy.yaml"


@dataclass(frozen=True)
class JobsConfig:  # typed settings for rules.jobs_demo in the policy
    max_idle_days: int = 90


class JobsScanner(Scanner):
    name = "jobs_demo"
    resource_types = ("job",)  # lets the run flag exclusions with a misspelled type
    config_model = JobsConfig
    rules = (
        RuleSpec(
            "jobs_demo.idle",  # must start with "<scanner name>."
            "Job has not run recently",
            Severity.MEDIUM,
            frozenset({Dimension.COST}),
            description="Jobs idle longer than max_idle_days may be candidates for review.",
            settings=("max_idle_days",),
            on_fail=Recommendation.REVIEW,
        ),
    )

    def discover(self, ctx: ScannerContext) -> Sequence[Resource]:
        # Demo data. A real scanner would read through ctx.get_sql() (read-only).
        source = "demo-data"
        return [
            Resource(
                ResourceRef("my-workspace", "job", "job-1"),
                name="nightly_etl",
                owner="team-data",
                evidence=(Evidence("days_idle", EvidenceStatus.OBSERVED, source, value=2),),
            ),
            Resource(
                ResourceRef("my-workspace", "job", "job-2"),
                name="old_export",
                evidence=(Evidence("days_idle", EvidenceStatus.UNAVAILABLE, source),),
            ),
        ]

    def analyze(
        self, ctx: ScannerContext, resources: Sequence[Resource]
    ) -> Sequence[RuleEvaluation]:
        evaluations = []
        for resource in resources:
            evidence = resource.get_evidence("days_idle")
            if evidence is None or evidence.status is EvidenceStatus.UNAVAILABLE:
                outcome, reason = Outcome.INDETERMINATE, "Run history unavailable."  # never guess
            elif evidence.value > ctx.config.max_idle_days:
                outcome, reason = Outcome.FAIL, f"Idle for {evidence.value} days."
            else:
                outcome, reason = Outcome.PASS, f"Ran {evidence.value} days ago."
            evaluations.append(
                RuleEvaluation(
                    resource.ref,
                    "jobs_demo.idle",
                    outcome,
                    evidence=(evidence,) if evidence else (),
                    reasons=(reason,),
                )
            )
        return evaluations

    def recommend(
        self, ctx: ScannerContext, scored: Sequence[ScoredEvaluation]
    ) -> Sequence[Finding]:
        return [
            Finding(
                resource=item.evaluation.resource,
                rule_id=item.evaluation.rule_id,
                severity=item.severity,
                recommendation=Recommendation.KEEP
                if item.evaluation.outcome is Outcome.PASS
                else Recommendation.REVIEW,
                reasons=item.evaluation.reasons,
                evidence=item.evaluation.evidence,
                confidence=item.confidence,
                impacts=frozenset({Dimension.COST}),
            )
            for item in scored
        ]

In [ ]:
display(rules_dataframe(spark, POLICY_PATH, scanners=["uc", JobsScanner]))

combined = run(POLICY_PATH, scanners=["uc", JobsScanner], spark=spark)
print(combined.summary())
display(combined.to_dataframe(spark))